# 🔬 ISIC 2020 – Melanoma Classification com PyTorch + GPU

**Objetivo:** Classificação binária de lesões de pele (benigno × maligno) usando o dataset ISIC 2020.  
**Aceleração:** GPU via CUDA / PyTorch (com AMP – Automatic Mixed Precision).  
**Seção 8 – Modelagem:** Três arquiteturas PyTorch comparadas — MLP tabular, ResNet-18 e EfficientNet-B0.

> Este notebook segue a estrutura do *Tech Challenge* FIAP Pós-Graduação 9AIDT Fase 01.

## 0. Instalação de dependências

In [ ]:
# Execute somente se as bibliotecas ainda não estiverem instaladas
import sys
!{sys.executable} -m pip install --quiet --upgrade pip
!{sys.executable} -m pip install --quiet matplotlib scikit-learn pandas seaborn tqdm
!{sys.executable} -m pip install --quiet albumentations timm
!{sys.executable} -m pip install --quiet torch torchvision --index-url https://download.pytorch.org/whl/cu121

## 1. Objetivo e contexto do problema

- Detectar automaticamente melanomas (malignos) entre lesões de pele em pacientes femininas.
- Apoiar decisão clínica — o modelo **não substitui** o diagnóstico médico.
- Avaliar três arquiteturas de redes neurais aceleradas por GPU para escolher a mais adequada ao cenário.

In [ ]:
objetivo = "Classificar lesões de pele (benigno x maligno) no dataset ISIC 2020 usando redes neurais em GPU."
contexto = "Suporte diagnóstico à saúde feminina com IA — pacientes do sexo feminino do ISIC 2020."
print("Objetivo:", objetivo)
print("Contexto:", contexto)

## 2. Importações e configuração

In [ ]:
import os, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_curve, auc,
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)

### 2.1 Configuração de GPU / CUDA

In [ ]:
# ── Detecção de GPU ───────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"          # AMP só faz sentido na GPU

print(f"Dispositivo      : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU              : {torch.cuda.get_device_name(0)}")
    print(f"VRAM total       : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"Versão CUDA      : {torch.version.cuda}")
    print(f"cuDNN habilitado : {torch.backends.cudnn.enabled}")
else:
    print("⚠️  CUDA não disponível — usando CPU (treinamento mais lento).")

print(f"PyTorch versão   : {torch.__version__}")
print(f"AMP habilitado   : {USE_AMP}")

### 2.2 Seed global e hiperparâmetros

In [ ]:
SEED = 42

def seed_everything(seed: int = SEED):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

# ── Caminhos ──────────────────────────────────────────────────────────────────
BASE_DIR   = Path("../data/isic2020")
TRAIN_DIR  = BASE_DIR / "Train"
TEST_DIR   = BASE_DIR / "test"
TRAIN_CSV  = BASE_DIR / "ISIC_2020_Training_GroundTruth.csv"
TEST_CSV   = BASE_DIR / "ISIC_2020_Test_Metadata.csv"
OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Hiperparâmetros comuns ────────────────────────────────────────────────────
IMG_SIZE     = 224
BATCH_SIZE   = 32
NUM_EPOCHS   = 10
LR           = 1e-4
WEIGHT_DECAY = 1e-5
FOLD_TO_TRAIN = 0
N_FOLDS      = 5
ACCUMULATE   = 2          # gradient accumulation steps (simula batch maior)
PATIENCE     = 3          # early stopping

# ── Colunas do dataset ────────────────────────────────────────────────────────
TARGET_COL  = "target"
IMG_ID_COL  = "image_name"
CLASS_NAMES = ["Benigno", "Maligno"]

print("Configuração concluída.")

## 3. Carregamento dos dados

Dados públicos do [ISIC 2020 Challenge](https://challenge2020.isic-archive.com/).  
Filtramos apenas pacientes do sexo feminino para o recorte de saúde da mulher.

In [ ]:
dataset_todos_train = pd.read_csv(TRAIN_CSV)
dataset_todos_teste = pd.read_csv(TEST_CSV)

df_train = dataset_todos_train[dataset_todos_train["sex"] == "female"].copy()
df_test  = dataset_todos_teste[dataset_todos_teste["sex"] == "female"].copy()

print(f"Treino : {len(df_train):,} registros")
print(f"Teste  : {len(df_test):,} registros")
print("\nColunas:", df_train.columns.tolist())
display(df_train.head())

## 4. Exploração de dados (EDA)

In [ ]:
display(df_train.describe(include="all").T)
print("\nValores nulos:")
print(df_train.isnull().sum())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

counts = df_train[TARGET_COL].value_counts()
axes[0].bar(CLASS_NAMES, counts.values, color=["steelblue", "tomato"])
axes[0].set_title("Distribuição de Classes")
axes[0].set_ylabel("Quantidade")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 20, f"{v:,}\n({v/len(df_train)*100:.1f}%)", ha="center")

if "age_approx" in df_train.columns:
    df_train.groupby(TARGET_COL)["age_approx"].hist(bins=20, ax=axes[1], alpha=0.6)
    axes[1].set_title("Distribuição de Idade por Classe")
    axes[1].legend(CLASS_NAMES)

if "anatom_site_general_challenge" in df_train.columns:
    site_counts = df_train.groupby(["anatom_site_general_challenge", TARGET_COL]).size().unstack(fill_value=0)
    site_counts.plot(kind="bar", ax=axes[2], color=["steelblue", "tomato"])
    axes[2].set_title("Localização anatômica por Classe")
    axes[2].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "eda_overview.png", dpi=100, bbox_inches="tight")
plt.show()

## 5. Pré-processamento

Pipeline reproduzível com `sklearn` para os metadados tabulares.

In [ ]:
FEATURE_COLS = ["age_approx", "anatom_site_general_challenge"]
NUMERIC_FEATURES = ["age_approx"]
CATEGORICAL_FEATURES = ["anatom_site_general_challenge"]

X = df_train[FEATURE_COLS].copy()
y = df_train[TARGET_COL].copy()

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, NUMERIC_FEATURES),
    ("cat", categorical_transformer, CATEGORICAL_FEATURES),
])

print("Preprocessor configurado.")

## 6. Análise de correlação

In [ ]:
corr_df = df_train.select_dtypes(include=["number"]).corr()
plt.figure(figsize=(10, 6))
sns.heatmap(corr_df, cmap="coolwarm", center=0, annot=True, fmt=".2f")
plt.title("Matriz de correlação (variáveis numéricas)")
plt.tight_layout()
plt.show()

## 7. Separação treino e teste

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
print(f"Treino: {X_train.shape}  |  Validação: {X_val.shape}")

# Pré-processar metadados
X_train_proc = preprocessor.fit_transform(X_train)
X_val_proc   = preprocessor.transform(X_val)
NUM_META = X_train_proc.shape[1]
print(f"Features após pré-processamento: {NUM_META}")

## 8. Modelagem

Nesta seção treinamos e comparamos **três arquiteturas PyTorch** aceleradas por GPU:

| # | Modelo | Entrada | Estratégia |
|---|--------|---------|------------|
| 1 | **MLP Tabular** | Metadados (idade + localização) | Rede totalmente conectada |
| 2 | **ResNet-18** | Imagens 224×224 + metadados | Transfer learning (ImageNet) |
| 3 | **EfficientNet-B0** | Imagens 224×224 + metadados | Transfer learning (ImageNet) |

Todos os modelos usam:
- `BCEWithLogitsLoss` com `pos_weight` para tratar desbalanceamento
- Otimizador `AdamW` com `CosineAnnealingLR`
- **AMP (Automatic Mixed Precision)** quando GPU disponível
- **Gradient Accumulation** para simular batch maior

### 8.1 Dataset e Augmentations (imagens)

In [ ]:
def get_transforms(phase: str, img_size: int = IMG_SIZE):
    if phase == "train":
        return A.Compose([
            A.RandomResizedCrop(size=(img_size, img_size), scale=(0.7, 1.0)),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=30, p=0.5),
            A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, p=0.4),
            A.CoarseDropout(max_holes=8, max_height=16, max_width=16, p=0.3),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2(),
        ])
    else:
        return A.Compose([
            A.Resize(img_size, img_size),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2(),
        ])


class ISICDataset(Dataset):
    """Dataset que combina imagens JPEG + metadados tabulares."""

    def __init__(self, df: pd.DataFrame, img_dir: Path, meta_array: np.ndarray,
                 transform=None):
        self.df        = df.reset_index(drop=True)
        self.img_dir   = Path(img_dir)
        self.meta      = meta_array.astype(np.float32)
        self.transform = transform

        # Filtra linhas cujas imagens existem
        mask = self.df[IMG_ID_COL].apply(
            lambda x: (self.img_dir / f"{x}.jpg").exists()
        )
        self.df   = self.df[mask].reset_index(drop=True)
        self.meta = self.meta[mask.values]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row    = self.df.iloc[idx]
        label  = torch.tensor(row[TARGET_COL], dtype=torch.float32)
        meta   = torch.tensor(self.meta[idx], dtype=torch.float32)
        img_path = self.img_dir / f"{row[IMG_ID_COL]}.jpg"
        image  = np.array(Image.open(img_path).convert("RGB"))
        if self.transform:
            image = self.transform(image=image)["image"]
        return image, meta, label


# ── DataLoaders para treino/val ───────────────────────────────────────────────
# Mantemos os metadados alinhados com df_train / df_val
meta_train_full = preprocessor.fit_transform(
    df_train[FEATURE_COLS]
)   # já foi feito na seção 7, mas refazemos aqui de forma explícita

ds_train = ISICDataset(
    df_train.iloc[X_train.index], TRAIN_DIR,
    meta_train_full[X_train.index], get_transforms("train")
)
ds_val = ISICDataset(
    df_train.iloc[X_val.index], TRAIN_DIR,
    meta_train_full[X_val.index], get_transforms("val")
)

loader_train = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=USE_AMP)
loader_val   = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=USE_AMP)

n_neg = (df_train.iloc[X_train.index][TARGET_COL] == 0).sum()
n_pos = (df_train.iloc[X_train.index][TARGET_COL] == 1).sum()
pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32).to(DEVICE)
print(f"pos_weight  : {pos_weight.item():.2f}")
print(f"Dataset treino: {len(ds_train):,}  |  Dataset val: {len(ds_val):,}")

### 8.2 Modelo 1 — MLP Tabular

In [ ]:
class MLPTabular(nn.Module):
    """Perceptron multi-camadas para metadados tabulares (sem imagem).

    Arquitetura:
        Input → FC(128) → BN → ReLU → Dropout(0.3)
               → FC(64)  → BN → ReLU → Dropout(0.3)
               → FC(1)   (logit)
    """

    def __init__(self, num_features: int, hidden: int = 128, drop: float = 0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(num_features, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(),
            nn.Dropout(drop),
            nn.Linear(hidden, hidden // 2),
            nn.BatchNorm1d(hidden // 2),
            nn.ReLU(),
            nn.Dropout(drop),
            nn.Linear(hidden // 2, 1),
        )

    def forward(self, images, meta):  # images ignorado neste modelo
        return self.net(meta).squeeze(1)


mlp_model = MLPTabular(num_features=NUM_META).to(DEVICE)
total_params = sum(p.numel() for p in mlp_model.parameters() if p.requires_grad)
print(f"[MLP] Parâmetros treináveis: {total_params:,}")

### 8.3 Modelo 2 — ResNet-18 com cabeça multimodal

In [ ]:
class ResNet18Classifier(nn.Module):
    """ResNet-18 pré-treinado (ImageNet) com fusão de metadados.

    Pipeline:
        Imagem → Backbone ResNet-18 (features 512-d) ┐
        Meta   → FC(32) → ReLU                       ├→ Concat → FC(1)
    """

    def __init__(self, num_meta: int, pretrained: bool = True, drop: float = 0.4):
        super().__init__()
        # Backbone
        backbone = timm.create_model("resnet18", pretrained=pretrained, num_classes=0)
        self.backbone = backbone
        feat_dim = backbone.num_features   # 512 para ResNet-18

        # Projeção dos metadados
        self.meta_fc = nn.Sequential(
            nn.Linear(num_meta, 32),
            nn.ReLU(),
        )

        # Cabeça final
        self.head = nn.Sequential(
            nn.Dropout(drop),
            nn.Linear(feat_dim + 32, 1),
        )

    def forward(self, images, meta):
        feat = self.backbone(images)          # (B, 512)
        meta_feat = self.meta_fc(meta)        # (B, 32)
        x = torch.cat([feat, meta_feat], dim=1)
        return self.head(x).squeeze(1)


resnet_model = ResNet18Classifier(num_meta=NUM_META).to(DEVICE)
total_params = sum(p.numel() for p in resnet_model.parameters() if p.requires_grad)
print(f"[ResNet-18] Parâmetros treináveis: {total_params:,}")

### 8.4 Modelo 3 — EfficientNet-B0 com cabeça multimodal

In [ ]:
class EfficientNetB0Classifier(nn.Module):
    """EfficientNet-B0 pré-treinado (ImageNet) com fusão de metadados.

    Mais leve e eficiente que variantes maiores; ótimo ponto de partida
    para comparação custo-benefício.

    Pipeline:
        Imagem → Backbone EfficientNet-B0 (features 1280-d) ┐
        Meta   → FC(32) → ReLU                              ├→ Concat → Dropout → FC(1)
    """

    def __init__(self, num_meta: int, pretrained: bool = True, drop: float = 0.4):
        super().__init__()
        backbone = timm.create_model("efficientnet_b0", pretrained=pretrained, num_classes=0)
        self.backbone = backbone
        feat_dim = backbone.num_features   # 1280 para EfficientNet-B0

        self.meta_fc = nn.Sequential(
            nn.Linear(num_meta, 32),
            nn.ReLU(),
        )
        self.head = nn.Sequential(
            nn.Dropout(drop),
            nn.Linear(feat_dim + 32, 1),
        )

    def forward(self, images, meta):
        feat = self.backbone(images)          # (B, 1280)
        meta_feat = self.meta_fc(meta)        # (B, 32)
        x = torch.cat([feat, meta_feat], dim=1)
        return self.head(x).squeeze(1)


effnet_model = EfficientNetB0Classifier(num_meta=NUM_META).to(DEVICE)
total_params = sum(p.numel() for p in effnet_model.parameters() if p.requires_grad)
print(f"[EfficientNet-B0] Parâmetros treináveis: {total_params:,}")

### 8.5 Funções de treino e validação com AMP

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, scaler,
                    accumulate: int = ACCUMULATE):
    model.train()
    running_loss = 0.0
    optimizer.zero_grad()

    pbar = tqdm(enumerate(loader), total=len(loader), desc="  train", leave=False)
    for step, (images, meta, labels) in pbar:
        images = images.to(DEVICE, non_blocking=True)
        meta   = meta.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with autocast(enabled=USE_AMP):
            logits = model(images, meta)
            loss   = criterion(logits, labels) / accumulate

        scaler.scale(loss).backward()

        if (step + 1) % accumulate == 0 or (step + 1) == len(loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        running_loss += loss.item() * accumulate

    return running_loss / len(loader)


@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    all_labels, all_probs = [], []

    for images, meta, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        meta   = meta.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with autocast(enabled=USE_AMP):
            logits = model(images, meta)
            loss   = criterion(logits, labels)

        probs = torch.sigmoid(logits)
        running_loss += loss.item()
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

    all_labels = np.array(all_labels)
    all_probs  = np.array(all_probs)
    val_auc    = roc_auc_score(all_labels, all_probs) if len(np.unique(all_labels)) > 1 else 0.0
    return running_loss / len(loader), val_auc, all_labels, all_probs


print("Funções de treino/validação definidas.")

### 8.6 Loop de treinamento comparativo

In [ ]:
def train_model(model, model_name: str, loader_tr, loader_vl,
                pos_weight, num_epochs: int = NUM_EPOCHS,
                lr: float = LR, patience: int = PATIENCE):
    """Treina um modelo e retorna o histórico + melhor AUC."""
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=lr * 0.01)
    scaler    = GradScaler(enabled=USE_AMP)

    ckpt_path  = OUTPUT_DIR / f"best_{model_name}.pth"
    history    = {"train_loss": [], "val_loss": [], "val_auc": []}
    best_auc   = 0.0
    patience_ct = 0

    for epoch in range(1, num_epochs + 1):
        tr_loss             = train_one_epoch(model, loader_tr, criterion, optimizer, scaler)
        vl_loss, vl_auc, _, _ = validate(model, loader_vl, criterion)
        scheduler.step()

        history["train_loss"].append(tr_loss)
        history["val_loss"].append(vl_loss)
        history["val_auc"].append(vl_auc)

        flag = ""
        if vl_auc > best_auc:
            best_auc = vl_auc
            torch.save({"model_state_dict": model.state_dict(), "epoch": epoch}, ckpt_path)
            flag = " ✅"
            patience_ct = 0
        else:
            patience_ct += 1

        print(f"[{model_name}] Época {epoch:02d}/{num_epochs} | "
              f"loss_tr={tr_loss:.4f}  loss_vl={vl_loss:.4f}  AUC={vl_auc:.4f}{flag}")

        if patience_ct >= patience:
            print(f"  Early stopping na época {epoch}.")
            break

    return history, best_auc, ckpt_path


# ─────────────────────────────────────────────────────────────────────────────
# Dicionário de modelos a comparar
# (Se não houver imagens disponíveis, o MLP usará apenas metadados tabulares)
# ─────────────────────────────────────────────────────────────────────────────
MODELS_TO_COMPARE = {
    "MLP_Tabular"      : MLPTabular(num_features=NUM_META).to(DEVICE),
    "ResNet18"         : ResNet18Classifier(num_meta=NUM_META).to(DEVICE),
    "EfficientNet_B0"  : EfficientNetB0Classifier(num_meta=NUM_META).to(DEVICE),
}

all_histories = {}
all_best_aucs = {}

for model_name, model in MODELS_TO_COMPARE.items():
    print(f"\n{'='*60}")
    print(f"  Treinando: {model_name}")
    print(f"{'='*60}")
    hist, best_auc, _ = train_model(
        model, model_name, loader_train, loader_val, pos_weight
    )
    all_histories[model_name] = hist
    all_best_aucs[model_name] = best_auc

print("\n✅ Treinamento de todos os modelos concluído.")

## 9. Avaliação e comparação dos modelos

Métricas: **Accuracy**, **Recall**, **F1-Score** e **AUC-ROC**.  
Em contexto médico, Recall e AUC são prioritários (minimizar falsos negativos).

In [ ]:
THRESHOLD = 0.5

comparison_rows = []
for model_name, model in MODELS_TO_COMPARE.items():
    ckpt = OUTPUT_DIR / f"best_{model_name}.pth"
    if ckpt.exists():
        model.load_state_dict(torch.load(ckpt, map_location=DEVICE)["model_state_dict"])
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    _, val_auc, y_true, y_prob = validate(model, loader_val, criterion)
    y_pred = (np.array(y_prob) >= THRESHOLD).astype(int)
    comparison_rows.append({
        "Modelo"   : model_name,
        "Accuracy" : accuracy_score(y_true, y_pred),
        "Recall"   : recall_score(y_true, y_pred, zero_division=0),
        "F1"       : f1_score(y_true, y_pred, zero_division=0),
        "AUC-ROC"  : val_auc,
    })
    print(f"\n=== {model_name} ===")
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))

results_df = pd.DataFrame(comparison_rows).sort_values("AUC-ROC", ascending=False)
print("\n📊 Resumo comparativo:")
display(results_df.set_index("Modelo").style.highlight_max(axis=0, color="lightgreen"))

In [ ]:
# ── Curvas de aprendizado por modelo ─────────────────────────────────────────
fig, axes = plt.subplots(len(all_histories), 2,
                         figsize=(14, 5 * len(all_histories)))
if len(all_histories) == 1:
    axes = [axes]

for ax_row, (model_name, hist) in zip(axes, all_histories.items()):
    epochs_ran = range(1, len(hist["train_loss"]) + 1)
    ax_row[0].plot(epochs_ran, hist["train_loss"], "b-o", label="Treino")
    ax_row[0].plot(epochs_ran, hist["val_loss"],   "r-o", label="Validação")
    ax_row[0].set_title(f"{model_name} — Loss")
    ax_row[0].set_xlabel("Época"); ax_row[0].set_ylabel("BCEWithLogitsLoss")
    ax_row[0].legend(); ax_row[0].grid(True)

    ax_row[1].plot(epochs_ran, hist["val_auc"], "g-o", label="Val AUC")
    ax_row[1].axhline(max(hist["val_auc"]), color="orange", linestyle="--",
                      label=f"Melhor AUC={max(hist['val_auc']):.4f}")
    ax_row[1].set_title(f"{model_name} — AUC-ROC")
    ax_row[1].set_xlabel("Época"); ax_row[1].set_ylabel("AUC-ROC")
    ax_row[1].legend(); ax_row[1].grid(True)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "learning_curves_comparison.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
# ── Gráfico de barras — AUC-ROC final ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
models_names = results_df["Modelo"].tolist()
aucs = results_df["AUC-ROC"].tolist()
bars = ax.bar(models_names, aucs, color=["steelblue", "darkorange", "mediumseagreen"])
ax.set_ylim(0, 1.05)
ax.set_ylabel("AUC-ROC")
ax.set_title("Comparação AUC-ROC — Validação")
for bar, val in zip(bars, aucs):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.01,
            f"{val:.4f}", ha="center", fontsize=11)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "auc_comparison.png", dpi=100, bbox_inches="tight")
plt.show()

## 10. Explicabilidade

Feature importance no MLP tabular e matriz de confusão do melhor modelo.

In [ ]:
# Seleciona o melhor modelo pelo AUC
best_model_name = max(all_best_aucs, key=all_best_aucs.get)
best_model = MODELS_TO_COMPARE[best_model_name]
ckpt = OUTPUT_DIR / f"best_{best_model_name}.pth"
if ckpt.exists():
    best_model.load_state_dict(torch.load(ckpt, map_location=DEVICE)["model_state_dict"])

print(f"Melhor modelo: {best_model_name} (AUC={all_best_aucs[best_model_name]:.4f})")

# Matriz de confusão do melhor modelo
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
_, _, y_true_best, y_prob_best = validate(best_model, loader_val, criterion)
y_pred_best = (np.array(y_prob_best) >= THRESHOLD).astype(int)

cm = confusion_matrix(y_true_best, y_pred_best)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"Matriz de Confusão — {best_model_name}")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrix_best.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
# Curva ROC do melhor modelo
fpr, tpr, thresholds = roc_curve(y_true_best, y_prob_best)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color="darkorange", lw=2, label=f"ROC (AUC = {roc_auc:.4f})")
plt.plot([0, 1], [0, 1], color="navy", lw=1, linestyle="--")
plt.xlabel("Taxa de Falsos Positivos")
plt.ylabel("Taxa de Verdadeiros Positivos")
plt.title(f"Curva ROC — {best_model_name}")
plt.legend(loc="lower right")
plt.grid(True)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "roc_curve_best.png", dpi=100, bbox_inches="tight")
plt.show()

## 11. Bloco EXTRA — Informações de GPU e uso de VRAM

In [ ]:
if DEVICE.type == "cuda":
    allocated = torch.cuda.max_memory_allocated(0) / 1e9
    reserved  = torch.cuda.max_memory_reserved(0)  / 1e9
    print(f"GPU            : {torch.cuda.get_device_name(0)}")
    print(f"VRAM total     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"VRAM alocada   : {allocated:.2f} GB")
    print(f"VRAM reservada : {reserved:.2f} GB")
    torch.cuda.empty_cache()
    print("Cache de GPU liberado.")
else:
    print("Executando em CPU.")

## 12. Discussão crítica e conclusão

**Pontos-chave:**

1. **Três modelos comparados** — O MLP Tabular depende apenas dos metadados (idade e localização anatômica), enquanto ResNet-18 e EfficientNet-B0 exploram também as imagens dermoscópicas, geralmente obtendo AUC superior.

2. **Desbalanceamento** — O dataset ISIC 2020 é altamente desbalanceado (~1-2% de casos malignos). Usamos `pos_weight` para penalizar mais os erros nos positivos e priorizamos **Recall** e **AUC-ROC**.

3. **AMP e GPU** — O uso de *Automatic Mixed Precision* (FP16) reduz o consumo de VRAM e acelera o treinamento ~1,5–2× em GPUs compatíveis sem perda perceptível de qualidade.

4. **Limitações** — O modelo é uma ferramenta de **apoio à decisão clínica**; a decisão final deve ser do dermatologista. Generalização para outras populações ou equipamentos requer validação adicional.

**Próximos passos sugeridos:**
- Aumentar o número de épocas e usar validação cruzada estratificada completa (K-Fold).
- Testar EfficientNet-B4 ou ConvNeXt para maior capacidade.
- Incorporar técnicas de explicabilidade (GradCAM) para visualizar regiões ativas nas imagens.

In [ ]:
conclusao = (
    "Comparamos três arquiteturas PyTorch (MLP, ResNet-18, EfficientNet-B0) aceleradas por GPU "
    "para classificação benigno × maligno no ISIC 2020. "
    f"O melhor modelo foi {best_model_name} com AUC-ROC = {all_best_aucs[best_model_name]:.4f}. "
    "O uso de GPU e AMP reduziu significativamente o tempo de treinamento. "
    "Este sistema deve ser utilizado apenas como suporte ao diagnóstico médico."
)
print(conclusao)